# Example 1.4.2 / 2: Logical Constraints

Companion notebook for **1.4.2 Range and Logical Constraints** in the AMPL lab Documentation, section "Logical Constraints". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Logical Constraints

With MP-based solvers (HiGHS, Gurobi, CPLEX, Xpress and others), constraints may combine
conditions with logical operators. AMPL and the solver turn them into linear constraints with
extra binary variables. Here at least one of `x` and `y` must be small. The model has no
data, so there is no `data.py`. `ampl.var["x"].value()` reads a scalar variable's value, and
the `Constraint` method `is_logical()` tells whether a constraint is a logical one:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0, <= 5;
var y >= 0, <= 5;
maximize Total: x + y;
subject to One_Small: x <= 1 or y <= 1;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["x"].value(), ampl.var["y"].value())
print(ampl.con["One_Small"].is_logical())

HiGHS 1.14.0: optimal solution; objective 6
0 simplex iterations
0 branching nodes
solved
1.0 5.0
True


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver highs;
solve;
display x, y;

HiGHS 1.14.0: optimal solution; objective 6
0 simplex iterations
0 branching nodes
x = 1
y = 5

